In [1]:
import os
import numpy as np
import pandas as pd

# TensorFlow Keras imports (compatible with installed tf_keras)
from tensorflow.keras.preprocessing.image import (
    load_img,
    img_to_array,
    ImageDataGenerator,
)
from tensorflow.keras.applications import VGG16
from tensorflow.keras import models, layers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Paths
train_path = "../input/train/train"
test_path = "../input/test/test"

# Load labels
label_frame = pd.read_csv("../input/train.csv")
test_frame = pd.read_csv("../input/sample_submission.csv")

# Prepare image arrays
x_train = []
x_test = []

# Load training images
for fname in label_frame["id"]:
    image_path = os.path.join(train_path, fname)
    pil_image = load_img(image_path, target_size=(32, 32, 3))
    np_image = img_to_array(pil_image)
    x_train.append(np_image)

# Load test images
for fname in test_frame["id"]:
    image_path = os.path.join(test_path, fname)
    pil_image = load_img(image_path, target_size=(32, 32, 3))
    np_image = img_to_array(pil_image)
    x_test.append(np_image)

# Convert to numpy arrays and normalize
x_train = np.array(x_train, dtype="float32") / 255.0
x_test = np.array(x_test, dtype="float32") / 255.0
y_train = np.array(label_frame["has_cactus"])

print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)



2026-05-06 12:12:54.253056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778069574.265594      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778069574.269484      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-06 12:12:54.284908: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Training data shape: (14175, 32, 32, 3)
Test data shape: (3325, 32, 32, 3)


In [2]:
# Data augmentation
augmentations = ImageDataGenerator(
    vertical_flip=True, horizontal_flip=True, zoom_range=0.1
)
augmentations.fit(x_train)




In [3]:
def get_model():
    # Base VGG16 model (pre‑trained on ImageNet)
    base = VGG16(include_top=False, weights="imagenet", input_shape=(32, 32, 3))
    base.trainable = True
    # Freeze early layers, fine‑tune from block5_conv3 onward
    set_trainable = False
    for layer in base.layers:
        if layer.name == "block5_conv3":
            set_trainable = True
        layer.trainable = set_trainable

    model = models.Sequential()
    model.add(base)
    model.add(layers.Flatten())
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(256, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model


model = get_model()



2026-05-06 12:13:08.052894: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778069588.053322      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:46:00.0, compute capability: 8.6


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [4]:
# Train the model using the current Keras API
batch_size = 64
steps_per_epoch = max(1, len(x_train) // batch_size)

model.fit(
    augmentations.flow(x_train, y_train, batch_size=batch_size),
    epochs=30,  # modest epochs to stay within runtime limits
    steps_per_epoch=steps_per_epoch,
    verbose=2,
)

# Predict on test set
y_predictions = model.predict(x_test, batch_size=batch_size)

# Prepare submission
submission = pd.DataFrame(
    {"id": test_frame["id"], "has_cactus": y_predictions.squeeze()}
)

submission.to_csv("submission.csv", index=False, columns=["id", "has_cactus"])
print("Submission file 'submission.csv' written successfully.")

Epoch 1/30


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1778069591.060367     107 service.cc:148] XLA service 0x7fc3d0012270 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778069591.060393     107 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-06 12:13:11.090938: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778069591.314966     107 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778069592.586450     107 device_compiler.h:188] Compile

221/221 - 7s - 34ms/step - accuracy: 0.9412 - loss: 0.1770
Epoch 2/30
221/221 - 0s - 75us/step - accuracy: 0.9688 - loss: 0.0947
Epoch 3/30


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


221/221 - 3s - 14ms/step - accuracy: 0.9674 - loss: 0.0967
Epoch 4/30
221/221 - 0s - 80us/step - accuracy: 0.9688 - loss: 0.0540
Epoch 5/30
221/221 - 3s - 14ms/step - accuracy: 0.9719 - loss: 0.0792
Epoch 6/30
221/221 - 0s - 85us/step - accuracy: 0.9375 - loss: 0.1617
Epoch 7/30
221/221 - 3s - 14ms/step - accuracy: 0.9733 - loss: 0.0763
Epoch 8/30
221/221 - 0s - 85us/step - accuracy: 0.9688 - loss: 0.0525
Epoch 9/30
221/221 - 3s - 14ms/step - accuracy: 0.9739 - loss: 0.0718
Epoch 10/30
221/221 - 0s - 84us/step - accuracy: 0.9688 - loss: 0.0489
Epoch 11/30
221/221 - 3s - 14ms/step - accuracy: 0.9792 - loss: 0.0594
Epoch 12/30
221/221 - 0s - 98us/step - accuracy: 0.9844 - loss: 0.0376
Epoch 13/30
221/221 - 3s - 14ms/step - accuracy: 0.9769 - loss: 0.0654
Epoch 14/30
221/221 - 0s - 83us/step - accuracy: 1.0000 - loss: 0.0133
Epoch 15/30
221/221 - 3s - 14ms/step - accuracy: 0.9782 - loss: 0.0611
Epoch 16/30
221/221 - 0s - 99us/step - accuracy: 1.0000 - loss: 0.0318
Epoch 17/30
221/221 - 3s